# Aromatase Classification Model Building (Colab GPU-Optimized) — 3-Tier Training

Train 14 classification models across 12 fingerprint types × 2 split strategies.

**3-Tier Training Strategy:** Models are split into fast/medium/slow tiers, each followed by an automatic zip download so you can retrieve partial results even if the runtime disconnects.

- **Tier 1 (Fast):** Ridge, Logistic Regression, Decision Tree, KNN, Naive Bayes, LDA
- **Tier 2 (Medium):** Gradient Boosting, XGBoost, AdaBoost, MLP
- **Tier 3 (Slow):** SVC (RBF), SVC (Linear), Random Forest

**Two training modes:**
1. **Default** (no class weighting) — 14 models × 12 FPs × 2 splits = 336 fits → `models_classification/`
2. **Balanced** (`class_weight="balanced"` for 7 supported models) → `models_classification_balanced/`

**Task:** Predict bioactivity class (3 classes: active, intermediate, inactive) from molecular fingerprints.

**Class definitions:**
- **Active**: pChEMBL > 7 (IC50 < 100 nM)
- **Intermediate**: 6 ≤ pChEMBL ≤ 7 (100 nM ≤ IC50 ≤ 1 μM)
- **Inactive**: pChEMBL < 6 (IC50 > 1 μM)

**Optimizations:**
- GPU acceleration via cuML (Logistic Regression, KNN, SVC, Random Forest) and XGBoost (`device="cuda"`)
- Parallel CV folds via `n_jobs=-1` for CPU-bound sklearn models
- Memory-efficient: float32 dtypes, sparse matrices for low-density binary FPs
- Incremental checkpointing (resumable)

**Runtime:** Select **G4 GPU** (RTX Pro 6000 Blackwell, 96GB VRAM) for best performance.  
Also works on T4/L4 with automatic fallback.

## 1. Setup & Install

In [ ]:
%%capture
# Install RAPIDS cuML for GPU-accelerated ML
# This works on Colab with T4, L4, A100, or G4 GPU runtimes
!pip install --extra-index-url=https://pypi.nvidia.com cuml-cu12 cudf-cu12 --quiet
!pip install xgboost tqdm --quiet

In [ ]:
import subprocess
import warnings
warnings.filterwarnings('ignore')

# Check GPU
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                           '--format=csv,noheader'], capture_output=True, text=True)
print(f"GPU: {gpu_info.stdout.strip()}")

# Check cuML availability
try:
    import cuml
    CUML_AVAILABLE = True
    print(f"cuML {cuml.__version__} available - GPU acceleration enabled")
except ImportError:
    CUML_AVAILABLE = False
    print("cuML not available - falling back to sklearn (CPU only)")

import xgboost
print(f"XGBoost {xgboost.__version__}")

## 2. Clone Repository & Configure Paths

In [ ]:
import os

# Clone the aromatase repository from GitHub
REPO_URL = 'https://github.com/dataprofessor/aromatase.git'
REPO_DIR = '/content/aromatase'

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

# Configure paths
SPLITS_DIR = os.path.join(REPO_DIR, 'data', 'splits')
CURATED_PATH = os.path.join(REPO_DIR, 'data', 'processed', 'aromatase_bioactivity_curated.csv')

# Two output directories: default (no weighting) and balanced (class_weight='balanced')
OUTPUT_DIR_DEFAULT = os.path.join(REPO_DIR, 'data', 'models_classification')
OUTPUT_DIR_BALANCED = os.path.join(REPO_DIR, 'data', 'models_classification_balanced')
os.makedirs(OUTPUT_DIR_DEFAULT, exist_ok=True)
os.makedirs(OUTPUT_DIR_BALANCED, exist_ok=True)

# Verify paths exist
assert os.path.isdir(SPLITS_DIR), f"Splits dir not found: {SPLITS_DIR}"
assert os.path.isfile(CURATED_PATH), f"Curated file not found: {CURATED_PATH}"

split_files = sorted(os.listdir(SPLITS_DIR))
print(f"Found {len(split_files)} split files in {SPLITS_DIR}")
print(f"Output (default): {OUTPUT_DIR_DEFAULT}")
print(f"Output (balanced): {OUTPUT_DIR_BALANCED}")

## 3. Data Loading Utilities

In [ ]:
import gc
import time

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

RANDOM_STATE = 42
N_FOLDS = 10

# Class encoding
CLASS_LABELS = ['inactive', 'intermediate', 'active']
CLASS_ENCODING = {'inactive': 0, 'intermediate': 1, 'active': 2}


def load_curated_targets(path):
    """Load bioactivity_class lookup from curated bioactivity data."""
    df = pd.read_csv(path, usecols=['molecule_chembl_id', 'bioactivity_class'])
    df = df.dropna(subset=['bioactivity_class'])
    # Keep first occurrence per molecule
    df = df.drop_duplicates(subset='molecule_chembl_id', keep='first')
    return df.set_index('molecule_chembl_id')['bioactivity_class'].map(CLASS_ENCODING).to_dict()


def load_split_data(fp_path, target_lookup, use_sparse=False):
    """Load fingerprint CSV and join with targets. Returns X, y.

    Parameters
    ----------
    fp_path : str
        Path to the split fingerprint CSV.
    target_lookup : dict
        molecule_chembl_id -> class label (0/1/2) mapping.
    use_sparse : bool
        If True and bit density < 30%, return sparse matrix.
    """
    df = pd.read_csv(fp_path, dtype={'molecule_chembl_id': str})

    # Float32 for all feature columns (halves memory)
    fp_cols = [c for c in df.columns if c != 'molecule_chembl_id']
    df[fp_cols] = df[fp_cols].astype(np.float32)

    # Join with targets
    df['bioactivity_class'] = df['molecule_chembl_id'].map(target_lookup)
    df = df.dropna(subset=['bioactivity_class'])

    X = df[fp_cols].values
    y = df['bioactivity_class'].values.astype(np.int32)

    # Use sparse if binary FP with low density
    if use_sparse:
        density = np.count_nonzero(X) / X.size
        if density < 0.30:
            X = csr_matrix(X)

    return X, y, fp_cols


# Preload target lookup (used for all fingerprints)
target_lookup = load_curated_targets(CURATED_PATH)
print(f"Loaded {len(target_lookup)} molecule targets")

# Show class distribution
class_counts = pd.Series(target_lookup).value_counts().sort_index()
print(f"\nClass distribution:")
for label, name in enumerate(CLASS_LABELS):
    count = class_counts.get(label, 0)
    print(f"  {name} ({label}): {count} ({count/len(target_lookup)*100:.1f}%)")
print(f"  Imbalance ratio: {class_counts.max()/class_counts.min():.1f}x")

## 4. Model Definitions (GPU-Aware)

In [ ]:
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, AdaBoostClassifier,
)
from sklearn.neural_network import MLPClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from xgboost import XGBClassifier


def get_models(use_gpu=True, balanced=False):
    """Return list of (name, model, supports_sparse) tuples.

    Parameters
    ----------
    use_gpu : bool
        Whether to use cuML GPU-accelerated models.
    balanced : bool
        If True, only returns the 7 models that support class_weight='balanced',
        with that parameter set. If False, returns all 14 models without class weighting.
    """
    gpu = use_gpu and CUML_AVAILABLE

    if gpu:
        from cuml.linear_model import LogisticRegression as cuLogistic
        from cuml.neighbors import KNeighborsClassifier as cuKNN
        from cuml.svm import SVC as cuSVC
        from cuml.ensemble import RandomForestClassifier as cuRF

    cw = 'balanced' if balanced else None

    # --- Models that support class_weight ---
    balanced_models = [
        ("Ridge Classifier",
         RidgeClassifier(alpha=1.0, class_weight=cw),
         False),

        ("Logistic Regression (L1)",
         cuLogistic(penalty='l1', C=1.0, max_iter=1000) if (gpu and not balanced)
         else LogisticRegression(penalty='l1', solver='saga', C=1.0,
                                 max_iter=1000, random_state=RANDOM_STATE, class_weight=cw),
         not gpu or balanced),

        ("Logistic Regression (ElasticNet)",
         LogisticRegression(penalty='elasticnet', solver='saga', C=1.0,
                            l1_ratio=0.5, max_iter=1000, random_state=RANDOM_STATE, class_weight=cw),
         False),

        ("SVC (RBF)",
         cuSVC(kernel='rbf', C=1.0) if (gpu and not balanced)
         else SVC(kernel='rbf', C=1.0, random_state=RANDOM_STATE, class_weight=cw),
         not gpu or balanced),

        ("SVC (Linear)",
         cuSVC(kernel='linear', C=1.0) if (gpu and not balanced)
         else SVC(kernel='linear', C=1.0, random_state=RANDOM_STATE, class_weight=cw),
         not gpu or balanced),

        ("Decision Tree",
         DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight=cw),
         True),

        ("Random Forest",
         cuRF(n_estimators=500, random_state=RANDOM_STATE) if (gpu and not balanced)
         else RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE,
                                     class_weight=cw, n_jobs=-1),
         not gpu or balanced),
    ]

    # --- Models that do NOT support class_weight ---
    other_models = [
        ("k-Nearest Neighbors",
         cuKNN(n_neighbors=5) if gpu else KNeighborsClassifier(n_neighbors=5),
         not gpu),

        ("Gradient Boosting",
         GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
         False),

        ("XGBoost",
         XGBClassifier(n_estimators=500, random_state=RANDOM_STATE,
                       device='cuda', tree_method='hist', verbosity=0,
                       objective='multi:softprob', num_class=3,
                       eval_metric='mlogloss'),
         True),

        ("AdaBoost",
         AdaBoostClassifier(n_estimators=500, random_state=RANDOM_STATE),
         False),

        ("MLP Classifier",
         MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=1000,
                       random_state=RANDOM_STATE, early_stopping=True),
         False),

        ("Gaussian Naive Bayes",
         GaussianNB(),
         False),

        ("Linear Discriminant Analysis",
         LinearDiscriminantAnalysis(),
         False),
    ]

    if balanced:
        # Only return the 7 models that support class_weight
        return balanced_models
    else:
        # Return all 14 models (no class weighting)
        return balanced_models + other_models


# Quick check
models_default = get_models(use_gpu=True, balanced=False)
models_balanced = get_models(use_gpu=True, balanced=True)
print(f"Default mode: {len(models_default)} models (no class weighting)")
print(f"Balanced mode: {len(models_balanced)} models (class_weight='balanced')")
print(f"\nBalanced models: {[n for n, _, _ in models_balanced]}")

## 5. Training Engine

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, matthews_corrcoef,
)
from sklearn.base import clone
from scipy.sparse import issparse
from tqdm.notebook import tqdm


def compute_metrics(y_true, y_pred):
    """Compute classification metrics: Accuracy, Balanced Accuracy, F1 (weighted), MCC."""
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    f1_w = f1_score(y_true, y_pred, average='weighted')
    mcc = matthews_corrcoef(y_true, y_pred)
    return acc, bal_acc, f1_w, mcc


def is_cuml_model(model):
    """Check if model is a cuML estimator."""
    module = getattr(model, '__module__', '') or ''
    return 'cuml' in module


def cuml_cross_val_predict(model, X, y, cv):
    """Manual cross_val_predict for cuML models (not sklearn-compatible CV)."""
    import cupy as cp

    predictions = np.zeros_like(y, dtype=np.int32)
    for train_idx, val_idx in cv.split(X, y):
        X_tr = cp.asarray(X[train_idx], dtype=cp.float32)
        y_tr = cp.asarray(y[train_idx], dtype=cp.int32)
        X_val = cp.asarray(X[val_idx], dtype=cp.float32)

        m = clone(model)
        m.fit(X_tr, y_tr)
        preds = m.predict(X_val)
        predictions[val_idx] = cp.asnumpy(preds).ravel().astype(np.int32)

    return predictions


def train_single_model(name, model, X_train, y_train, X_test, y_test,
                        cv, supports_sparse):
    """Train a single model, compute train/CV/test metrics. Returns dict."""
    start = time.time()

    # Convert sparse to dense for models that don't support it
    X_tr = X_train.toarray() if issparse(X_train) and not supports_sparse else X_train
    X_te = X_test.toarray() if issparse(X_test) and not supports_sparse else X_test

    # For cuML models, ensure dense float32 numpy arrays
    if is_cuml_model(model):
        X_tr = np.ascontiguousarray(X_tr if not issparse(X_tr) else X_tr.toarray(), dtype=np.float32)
        X_te = np.ascontiguousarray(X_te if not issparse(X_te) else X_te.toarray(), dtype=np.float32)
        y_tr = np.ascontiguousarray(y_train, dtype=np.int32)
    else:
        y_tr = y_train

    # Fit
    model.fit(X_tr, y_tr)

    # Predict train & test
    y_pred_train = model.predict(X_tr)
    y_pred_test = model.predict(X_te)

    # Ensure numpy arrays (cuML may return cupy)
    try:
        import cupy as cp
        if isinstance(y_pred_train, cp.ndarray):
            y_pred_train = cp.asnumpy(y_pred_train)
        if isinstance(y_pred_test, cp.ndarray):
            y_pred_test = cp.asnumpy(y_pred_test)
    except ImportError:
        pass

    y_pred_train = np.asarray(y_pred_train).ravel().astype(np.int32)
    y_pred_test = np.asarray(y_pred_test).ravel().astype(np.int32)

    # Cross-validation (Stratified)
    if is_cuml_model(model):
        y_pred_cv = cuml_cross_val_predict(clone(model), X_tr, y_tr, cv)
    else:
        # Use n_jobs=-1 for CPU parallelism on sklearn models
        model_cv = clone(model)
        y_pred_cv = cross_val_predict(model_cv, X_tr, y_tr, cv=cv, n_jobs=-1)

    y_pred_cv = np.asarray(y_pred_cv).ravel().astype(np.int32)

    elapsed = time.time() - start

    # Metrics
    acc_train, bal_acc_train, f1_train, mcc_train = compute_metrics(y_train, y_pred_train)
    acc_cv, bal_acc_cv, f1_cv, mcc_cv = compute_metrics(y_train, y_pred_cv)
    acc_test, bal_acc_test, f1_test, mcc_test = compute_metrics(y_test, y_pred_test)

    return {
        'train': {'model': name, 'accuracy': acc_train, 'balanced_accuracy': bal_acc_train,
                  'f1_weighted': f1_train, 'mcc': mcc_train, 'time_s': elapsed},
        'cv': {'model': name, 'accuracy': acc_cv, 'balanced_accuracy': bal_acc_cv,
               'f1_weighted': f1_cv, 'mcc': mcc_cv, 'time_s': elapsed},
        'test': {'model': name, 'accuracy': acc_test, 'balanced_accuracy': bal_acc_test,
                 'f1_weighted': f1_test, 'mcc': mcc_test, 'time_s': elapsed},
    }

## 6. Main Training Loop (All Fingerprints × Splits × Both Modes)

In [ ]:
# Discover all fingerprint/split combinations
FINGERPRINTS = [
    'maccs', 'pubchem', 'substructure', 'klekota_roth',
    'cdk', 'cdk_ext', 'cdk_graphonly', 'estate',
    'atompairs2d', 'substructure_count', 'klekota_roth_count', 'atompairs2d_count',
]
SPLITS = ['random', 'kennard_stone']

# Binary fingerprints benefit from sparse storage
BINARY_FPS = {'maccs', 'pubchem', 'substructure', 'klekota_roth',
              'cdk', 'cdk_ext', 'cdk_graphonly', 'estate', 'atompairs2d'}

CSV_FIELDS = ['model', 'accuracy', 'balanced_accuracy', 'f1_weighted', 'mcc', 'time_s']


def get_split_paths(fp_name, split_name):
    """Resolve train/test file paths for a fingerprint+split combination."""
    stem = f"aromatase_{fp_name}_fp"
    train = os.path.join(SPLITS_DIR, f"{stem}_{split_name}_train.csv")
    test = os.path.join(SPLITS_DIR, f"{stem}_{split_name}_test.csv")
    return train, test


def load_completed(path):
    """Return set of model names already saved."""
    if not os.path.exists(path):
        return set()
    df = pd.read_csv(path)
    return set(df['model'].tolist())


def append_result(path, row_dict):
    """Append a single result row to CSV (creates file if needed)."""
    file_exists = os.path.exists(path) and os.path.getsize(path) > 0
    df = pd.DataFrame([row_dict], columns=CSV_FIELDS)
    df.to_csv(path, mode='a', header=not file_exists, index=False)


def run_training(output_dir, models_list, mode_label):
    """Run training loop for a given model list and output directory."""
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    total_combos = len(FINGERPRINTS) * len(SPLITS)
    combo_pbar = tqdm(total=total_combos, desc=f"{mode_label} FP×Split", position=0)

    results = []

    for fp_name in FINGERPRINTS:
        for split_name in SPLITS:
            train_path, test_path = get_split_paths(fp_name, split_name)
            if not os.path.exists(train_path):
                combo_pbar.update(1)
                continue

            # Output file paths
            prefix = f"{fp_name}_{split_name}_split"
            out_train = os.path.join(output_dir, f"{prefix}_train.csv")
            out_cv = os.path.join(output_dir, f"{prefix}_cv.csv")
            out_test = os.path.join(output_dir, f"{prefix}_test.csv")

            # Check what's already done
            completed = load_completed(out_train)
            models_to_run = [(n, m, sp) for n, m, sp in models_list if n not in completed]

            if not models_to_run:
                combo_pbar.update(1)
                continue

            # Load data
            use_sparse = fp_name in BINARY_FPS
            X_train, y_train, fp_cols = load_split_data(train_path, target_lookup, use_sparse)
            X_test, y_test, _ = load_split_data(test_path, target_lookup, use_sparse)

            desc = f"{fp_name}/{split_name}"
            model_pbar = tqdm(models_to_run, desc=desc, position=1, leave=False)

            for name, model, supports_sparse in model_pbar:
                model_pbar.set_postfix_str(name)
                try:
                    result = train_single_model(
                        name, clone(model), X_train, y_train, X_test, y_test,
                        cv, supports_sparse
                    )
                    # Checkpoint immediately
                    append_result(out_train, result['train'])
                    append_result(out_cv, result['cv'])
                    append_result(out_test, result['test'])

                    results.append({
                        'fingerprint': fp_name, 'split': split_name,
                        **result['test']
                    })
                except Exception as e:
                    print(f"\n  ERROR: {fp_name}/{split_name}/{name}: {e}")
                    continue

            # Free memory after each fingerprint
            del X_train, y_train, X_test, y_test
            gc.collect()

            combo_pbar.update(1)

    combo_pbar.close()
    print(f"\n{mode_label} done! Results saved to: {output_dir}")
    return results


# Verify all split files exist
missing = []
for fp in FINGERPRINTS:
    for split in SPLITS:
        train_p, test_p = get_split_paths(fp, split)
        if not os.path.exists(train_p):
            missing.append(f"{fp}/{split}")

if missing:
    print(f"WARNING: Missing split files for: {missing}")
else:
    n_default = len(FINGERPRINTS) * len(SPLITS) * len(get_models(use_gpu=True, balanced=False))
    n_balanced = len(FINGERPRINTS) * len(SPLITS) * len(get_models(use_gpu=True, balanced=True))
    print(f"All {len(FINGERPRINTS) * len(SPLITS)} fingerprint×split combinations found.")
    print(f"Default mode: {n_default} model fits")
    print(f"Balanced mode: {n_balanced} model fits")
    print(f"Total: {n_default + n_balanced} model fits (+ {N_FOLDS}-fold stratified CV each)")

### Tier 1: Fast Models

In [ ]:
# === TIER 1: FAST MODELS ===
TIER1_MODELS = ['Ridge Classifier', 'Logistic Regression (L1)', 'Logistic Regression (ElasticNet)', 'Decision Tree', 'k-Nearest Neighbors', 'Gaussian Naive Bayes', 'Linear Discriminant Analysis']

models_default_tier1 = [(n, m, sp) for n, m, sp in get_models(True, False) if n in TIER1_MODELS]
models_balanced_tier1 = [(n, m, sp) for n, m, sp in get_models(True, True) if n in TIER1_MODELS]

print("=" * 70)
print(f"TIER 1 (Fast) — Default mode ({len(models_default_tier1)} models)")
print("=" * 70)
results_default_tier1 = run_training(OUTPUT_DIR_DEFAULT, models_default_tier1, "Default-Fast")

if models_balanced_tier1:
    print("\n" + "=" * 70)
    print(f"TIER 1 (Fast) — Balanced mode ({len(models_balanced_tier1)} models)")
    print("=" * 70)
    results_balanced_tier1 = run_training(OUTPUT_DIR_BALANCED, models_balanced_tier1, "Balanced-Fast")
else:
    print(f"\nNo balanced-mode models in Tier 1.")


In [ ]:
# === Download Tier 1 (Fast) results as ZIP ===
import zipfile
from google.colab import files

zip_name = 'classification_tier1_fast.zip'

print(f"Zipping results from:")
print(f"  Default: {OUTPUT_DIR_DEFAULT}")
print(f"  Balanced: {OUTPUT_DIR_BALANCED}")

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for dir_path in [OUTPUT_DIR_DEFAULT, OUTPUT_DIR_BALANCED]:
        if os.path.isdir(dir_path):
            for root, _, filenames in os.walk(dir_path):
                for fname in sorted(filenames):
                    fpath = os.path.join(root, fname)
                    arcname = os.path.relpath(fpath, REPO_DIR)
                    zf.write(fpath, arcname)

    print(f"\nContents ({len(zf.infolist())} files):")
    for info in zf.infolist():
        print(f"  {info.filename} ({info.file_size/1024:.1f} KB)")

print(f"\n{zip_name} ({os.path.getsize(zip_name)/1024:.1f} KB)")
files.download(zip_name)

### Tier 2: Medium Models

In [ ]:
# === TIER 2: MEDIUM MODELS ===
TIER2_MODELS = ['Gradient Boosting', 'XGBoost', 'AdaBoost', 'MLP Classifier']

models_default_tier2 = [(n, m, sp) for n, m, sp in get_models(True, False) if n in TIER2_MODELS]
models_balanced_tier2 = [(n, m, sp) for n, m, sp in get_models(True, True) if n in TIER2_MODELS]

print("=" * 70)
print(f"TIER 2 (Medium) — Default mode ({len(models_default_tier2)} models)")
print("=" * 70)
results_default_tier2 = run_training(OUTPUT_DIR_DEFAULT, models_default_tier2, "Default-Medium")

if models_balanced_tier2:
    print("\n" + "=" * 70)
    print(f"TIER 2 (Medium) — Balanced mode ({len(models_balanced_tier2)} models)")
    print("=" * 70)
    results_balanced_tier2 = run_training(OUTPUT_DIR_BALANCED, models_balanced_tier2, "Balanced-Medium")
else:
    print(f"\nNo balanced-mode models in Tier 2.")


In [ ]:
# === Download Tier 2 (Fast + Medium) results as ZIP ===
import zipfile
from google.colab import files

zip_name = 'classification_tier2_medium.zip'

print(f"Zipping results from:")
print(f"  Default: {OUTPUT_DIR_DEFAULT}")
print(f"  Balanced: {OUTPUT_DIR_BALANCED}")

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for dir_path in [OUTPUT_DIR_DEFAULT, OUTPUT_DIR_BALANCED]:
        if os.path.isdir(dir_path):
            for root, _, filenames in os.walk(dir_path):
                for fname in sorted(filenames):
                    fpath = os.path.join(root, fname)
                    arcname = os.path.relpath(fpath, REPO_DIR)
                    zf.write(fpath, arcname)

    print(f"\nContents ({len(zf.infolist())} files):")
    for info in zf.infolist():
        print(f"  {info.filename} ({info.file_size/1024:.1f} KB)")

print(f"\n{zip_name} ({os.path.getsize(zip_name)/1024:.1f} KB)")
files.download(zip_name)

### Tier 3: Slow Models

In [ ]:
# === TIER 3: SLOW MODELS ===
TIER3_MODELS = ['SVC (RBF)', 'SVC (Linear)', 'Random Forest']

models_default_tier3 = [(n, m, sp) for n, m, sp in get_models(True, False) if n in TIER3_MODELS]
models_balanced_tier3 = [(n, m, sp) for n, m, sp in get_models(True, True) if n in TIER3_MODELS]

print("=" * 70)
print(f"TIER 3 (Slow) — Default mode ({len(models_default_tier3)} models)")
print("=" * 70)
results_default_tier3 = run_training(OUTPUT_DIR_DEFAULT, models_default_tier3, "Default-Slow")

if models_balanced_tier3:
    print("\n" + "=" * 70)
    print(f"TIER 3 (Slow) — Balanced mode ({len(models_balanced_tier3)} models)")
    print("=" * 70)
    results_balanced_tier3 = run_training(OUTPUT_DIR_BALANCED, models_balanced_tier3, "Balanced-Slow")
else:
    print(f"\nNo balanced-mode models in Tier 3.")


In [ ]:
# === Download ALL results (Tier 1 + 2 + 3) results as ZIP ===
import zipfile
from google.colab import files

zip_name = 'classification_all_results.zip'

print(f"Zipping results from:")
print(f"  Default: {OUTPUT_DIR_DEFAULT}")
print(f"  Balanced: {OUTPUT_DIR_BALANCED}")

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for dir_path in [OUTPUT_DIR_DEFAULT, OUTPUT_DIR_BALANCED]:
        if os.path.isdir(dir_path):
            for root, _, filenames in os.walk(dir_path):
                for fname in sorted(filenames):
                    fpath = os.path.join(root, fname)
                    arcname = os.path.relpath(fpath, REPO_DIR)
                    zf.write(fpath, arcname)

    print(f"\nContents ({len(zf.infolist())} files):")
    for info in zf.infolist():
        print(f"  {info.filename} ({info.file_size/1024:.1f} KB)")

print(f"\n{zip_name} ({os.path.getsize(zip_name)/1024:.1f} KB)")
files.download(zip_name)

## 7. Results Aggregation

In [ ]:
# Load all test results from BOTH modes into master DataFrames

def load_all_results(output_dir, mode_label):
    """Load all test results from an output directory."""
    all_results = []
    for fp_name in FINGERPRINTS:
        for split_name in SPLITS:
            prefix = f"{fp_name}_{split_name}_split"
            test_path = os.path.join(output_dir, f"{prefix}_test.csv")
            if os.path.exists(test_path):
                df = pd.read_csv(test_path)
                df['fingerprint'] = fp_name
                df['split'] = split_name
                df['mode'] = mode_label
                all_results.append(df)
    return all_results


results_default_list = load_all_results(OUTPUT_DIR_DEFAULT, 'default')
results_balanced_list = load_all_results(OUTPUT_DIR_BALANCED, 'balanced')

all_test_results = results_default_list + results_balanced_list

if all_test_results:
    results_df = pd.concat(all_test_results, ignore_index=True)
    print(f"Total results: {len(results_df)}")
    print(f"  Default mode: {len([r for r in results_default_list for _ in [1]])} files")
    print(f"  Balanced mode: {len([r for r in results_balanced_list for _ in [1]])} files")
    print(f"\nTop 20 models by test Balanced Accuracy (across both modes):")
    top20 = results_df.nlargest(20, 'balanced_accuracy')[
        ['mode', 'fingerprint', 'split', 'model', 'accuracy', 'balanced_accuracy', 'f1_weighted', 'mcc', 'time_s']
    ]
    display(top20.reset_index(drop=True))
else:
    print("No results found yet. Run the training loop first.")

In [ ]:
# Best model per fingerprint — compare default vs balanced
if all_test_results:
    best_per_fp = results_df.loc[
        results_df.groupby(['mode', 'fingerprint', 'split'])['balanced_accuracy'].idxmax()
    ][['mode', 'fingerprint', 'split', 'model', 'balanced_accuracy', 'f1_weighted', 'mcc']].sort_values(
        'balanced_accuracy', ascending=False
    )
    print("Best model per fingerprint × split × mode (by balanced accuracy):")
    display(best_per_fp.reset_index(drop=True))

    # --- Direct comparison: same models default vs balanced ---
    print("\n" + "=" * 70)
    print("COMPARISON: Default vs Balanced (same 7 models that support class_weight)")
    print("=" * 70)
    balanced_model_names = [n for n, _, _ in get_models(use_gpu=False, balanced=True)]
    comparison_df = results_df[results_df['model'].isin(balanced_model_names)].copy()
    comp_pivot = comparison_df.pivot_table(
        index=['model', 'fingerprint', 'split'],
        columns='mode',
        values='balanced_accuracy'
    ).reset_index()
    if 'default' in comp_pivot.columns and 'balanced' in comp_pivot.columns:
        comp_pivot['improvement'] = comp_pivot['balanced'] - comp_pivot['default']
        print(f"\nMean improvement (balanced - default): {comp_pivot['improvement'].mean():.4f}")
        print(f"Models improved: {(comp_pivot['improvement'] > 0).sum()}/{len(comp_pivot)}")
        print(f"\nPer-model mean improvement:")
        display(comp_pivot.groupby('model')['improvement'].mean().sort_values(ascending=False).round(4))

## 8. Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if all_test_results:
    for mode in ['default', 'balanced']:
        mode_df = results_df[results_df['mode'] == mode]
        if mode_df.empty:
            continue

        for split_name, split_label in [('random', 'Random'), ('kennard_stone', 'Kennard-Stone')]:
            split_df = mode_df[mode_df['split'] == split_name]
            if split_df.empty:
                continue

            pivot = split_df.pivot_table(index='model', columns='fingerprint', values='balanced_accuracy')
            pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

            fig, ax = plt.subplots(figsize=(14, 8))
            sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', center=0.5,
                        vmin=0.3, vmax=0.9, ax=ax, linewidths=0.5)
            title = f'Test Balanced Accuracy — {split_label} Split ({mode.title()} mode)'
            ax.set_title(title, fontsize=13)
            ax.set_xlabel('Fingerprint')
            ax.set_ylabel('Model')
            plt.tight_layout()
            fname = f'heatmap_balanced_acc_{split_name}_{mode}.png'
            plt.savefig(os.path.join(OUTPUT_DIR_DEFAULT, fname), dpi=150, bbox_inches='tight')
            plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

if all_test_results:
    # --- Confusion matrix for the best overall model ---
    best_row = results_df.loc[results_df['balanced_accuracy'].idxmax()]
    best_fp = best_row['fingerprint']
    best_split = best_row['split']
    best_model_name = best_row['model']
    best_mode = best_row['mode']

    print(f"Best model: {best_model_name} ({best_fp}, {best_split}, {best_mode} mode)")
    print(f"Balanced Accuracy: {best_row['balanced_accuracy']:.4f}")
    print(f"F1 (weighted): {best_row['f1_weighted']:.4f}")
    print(f"MCC: {best_row['mcc']:.4f}")

    # Retrain best model to get predictions for confusion matrix
    train_path, test_path = get_split_paths(best_fp, best_split)
    use_sparse = best_fp in BINARY_FPS
    X_train, y_train, _ = load_split_data(train_path, target_lookup, use_sparse)
    X_test, y_test, _ = load_split_data(test_path, target_lookup, use_sparse)

    # Get the model (use balanced=True if best was from balanced mode)
    use_balanced = (best_mode == 'balanced')
    all_models_list = get_models(use_gpu=False, balanced=use_balanced)
    best_model = None
    for name, m, _ in all_models_list:
        if name == best_model_name:
            best_model = clone(m)
            break

    if best_model is not None:
        if issparse(X_train):
            X_train = X_train.toarray()
            X_test = X_test.toarray()

        best_model.fit(X_train, y_train)
        y_pred = best_model.predict(X_test)

        # Confusion matrix
        fig, ax = plt.subplots(figsize=(8, 6))
        cm = confusion_matrix(y_test, y_pred)
        disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_LABELS)
        disp.plot(ax=ax, cmap='Blues', values_format='d')
        ax.set_title(f'Confusion Matrix — {best_model_name}\n({best_fp}, {best_split}, {best_mode})', fontsize=12)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR_DEFAULT, 'confusion_matrix_best.png'), dpi=150, bbox_inches='tight')
        plt.show()

        # Classification report
        print(f"\nClassification Report:")
        print(classification_report(y_test, y_pred, target_names=CLASS_LABELS, digits=4))

In [ ]:
if all_test_results:
    # --- Runtime comparison (default mode, random split) ---
    default_random = results_df[(results_df['mode'] == 'default') & (results_df['split'] == 'random')]
    if not default_random.empty:
        time_df = default_random.groupby('model')['time_s'].mean().sort_values()

        fig, ax = plt.subplots(figsize=(10, 6))
        time_df.plot(kind='barh', ax=ax, color='steelblue')
        ax.set_xlabel('Average time per model (seconds)')
        ax.set_title('Model Training Time (avg across fingerprints, random split, default mode)')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR_DEFAULT, 'runtime_comparison.png'), dpi=150, bbox_inches='tight')
        plt.show()

    # --- Random vs Kennard-Stone comparison (default mode) ---
    default_df = results_df[results_df['mode'] == 'default']
    comparison = default_df.groupby(['model', 'split'])['balanced_accuracy'].mean().unstack()
    if 'kennard_stone' in comparison.columns:
        comparison['diff'] = comparison['random'] - comparison['kennard_stone']
        comparison = comparison.sort_values('random', ascending=False)
        print("\nRandom vs Kennard-Stone (mean Balanced Accuracy, default mode):")
        display(comparison.round(4))

In [ ]:
if all_test_results:
    # Save master results table (both modes combined)
    master_path = os.path.join(OUTPUT_DIR_DEFAULT, 'all_results_master.csv')
    results_df.to_csv(master_path, index=False)
    print(f"Master results saved to: {master_path}")
    print(f"Shape: {results_df.shape}")
    print(f"Modes: {results_df['mode'].value_counts().to_dict()}")